Practice problems by topic

Each problem includes a short hint. Ask for solutions when ready.
SELECT (basic -> intermediate)

List all columns for every employee.

Hint: simple SELECT * FROM employees;
Return the full name (first + last) and salary for all employees, ordered by salary descending.

Hint: string concatenation or use database-specific function (e.g., CONCAT).
Show department name and number of employees in each department (include departments with zero employees).

Hint: LEFT JOIN departments to employees and GROUP BY department fields.
For the orders table, show each distinct product and the total revenue (sum of amount) per product.

Hint: GROUP BY product and use SUM(amount).
Filtering (WHERE, AND/OR, LIKE, IN, BETWEEN) 5) Find active employees with salary greater than $60,000.

Hint: WHERE status='active' AND salary > 60000
Find employees hired between 2019-01-01 and 2022-12-31.

Hint: WHERE hire_date BETWEEN '2019-01-01' AND '2022-12-31'
Find orders for customers whose name contains 'Acme' or 'Beta' (case-insensitive).

Hint: Use LIKE '%Acme%' or use IN for exact matches; consider ILIKE in some DBs.
Return employees who are not assigned to any department.

Hint: WHERE department_id IS NULL
NULL handling (IS NULL, IS NOT NULL, COALESCE, NULLIF, COUNT vs COUNT(column)) 9) List employees and show salary where missing salary is displayed as 0.

Hint: COALESCE(salary, 0) or ISNULL depending on DB.
Count how many employees have a recorded performance_score and how many do not.

Hint: COUNT(performance_score) vs COUNT(*) or use SUM(CASE WHEN performance_score IS NULL THEN 1 ELSE 0 END).
For orders, list order_id and a column 'status' that shows:

'Canceled' if canceled = TRUE
'Shipped' if shipped_date IS NOT NULL and canceled is FALSE
'Pending' otherwise
Hint: NULL values in canceled mean pending; use CASE (covered next) or COALESCE.
CASE expressions (conditional logic) 12) Create a salary_band column for employees: - 'Low' for salary < 50000 - 'Mid' for salary BETWEEN 50000 AND 90000 - 'High' for salary > 90000 - 'Unknown' if salary IS NULL - Hint: Use CASE WHEN ... THEN ... ELSE ...

Show employees and a column 'tenure' using CASE:

'<1 year' if hire_date within last 365 days
'1-3 years'
'3+ years'
Hint: Use current_date or a fixed date for reproducible results; use DATE_DIFF or YEAR difference depending on DB.
Using CASE, create a column 'performance_label':

1-3 => 'Poor', 4-6 => 'Average', 7-8 => 'Good', 9-10 => 'Excellent', NULL => 'Not Reviewed'
Hint: Chain WHEN conditions and handle NULL explicitly.
Data types and conversion (CAST/CONVERT, formatting, arithmetic) 15) Show order_id and order_date formatted as 'YYYY-MM-DD' and as a year integer. - Hint: Use CAST(EXTRACT(YEAR FROM order_date) AS INT) or database equivalent.

Calculate total payroll per department (sum of salary), but treat NULL salaries as 0. Show department name and total payroll as DECIMAL with two decimals.

Hint: COALESCE(salary,0), GROUP BY department.
Return order_id and amount in integer dollars (truncate cents), and create a column rounded_amount (rounded to nearest integer).

Hint: Use CAST(amount AS INT) or FLOOR(), and ROUND(amount).
Example of NULL pitfalls:

Explain why this returns unexpected results: WHERE department_id NOT IN (1,2,3)
Hint: If department_id can be NULL, NOT IN may behave unexpectedly because NULL comparisons make the predicate UNKNOWN. Use WHERE (department_id NOT IN (... ) OR department_id IS NULL) or use NOT EXISTS subquery.
Combined challenge problems (use multiple topics) 19) For each department show: - department name, - number of active employees, - average salary of active employees (exclude NULL salaries), - highest performance_score among active employees, - default the average salary to 0 if no active employees. - Hint: LEFT JOIN + aggregation + COALESCE + WHERE inside aggregate via CASE WHEN status='active' THEN salary END

Identify "at-risk" employees using a CASE:

An employee is 'At Risk' if performance_score <= 4 OR salary < 50000 OR status='on_leave'
Else 'OK' (unknown scores count as 'At Risk')
Show employee_id, full name, salary, performance_score, and the at-risk label.
Hint: Be explicit with NULL checks (e.g., performance_score IS NULL should map to 'At Risk').
For orders produce a summary per customer:

customer_name, total_orders, total_amount, last_order_date, pending_orders_count (pending: shipped_date IS NULL AND canceled IS NOT TRUE)
Hint: Use GROUP BY customer_name and conditional aggregation: SUM(CASE WHEN ... THEN 1 ELSE 0 END).
Extra tips and best practices

Prefer explicit column lists in SELECT instead of SELECT * in production.
Be careful mixing NULLs with IN/NOT IN and equality comparisons.
Use COALESCE to provide defaults for NULLs; use NULLIF to avoid divide-by-zero or to convert a sentinel value to NULL.
When casting/formatting dates and numbers, be mindful of your RDBMS functions (FORMAT, TO_CHAR, CONVERT, CAST).
Use CASE for readable conditional logic; use boolean short-circuits carefully.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS sql;
CREATE SCHEMA IF NOT EXISTS sql.practice;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sql.practice.departments (
  department_id INT PRIMARY KEY,
  name VARCHAR(50) NOT NULL,
  budget DECIMAL(12,2) NOT NULL,
  location VARCHAR(50) 
);


In [0]:
%sql
INSERT INTO sql.practice.departments (department_id, name, budget, location) VALUES
(1, 'Engineering', 1200000.00, 'Austin'),
(2, 'Sales', 600000.00, 'New York'),
(3, 'HR', 200000.00, NULL),
(4, 'Finance', 350000.00, 'Chicago');

In [0]:
%sql
SELECT* FROM sql.practice.departments;

In [0]:
%sql
-- Employees table
CREATE TABLE IF NOT EXISTS sql.practice.employees (
  employee_id INT PRIMARY KEY,
  first_name VARCHAR(30) NOT NULL,
  last_name VARCHAR(30) NOT NULL,
  department_id INT,
  hire_date DATE NOT NULL,
  salary DECIMAL(10,2) NOT NULL,
  manager_id INT,
  status VARCHAR(20) NOT NULL, -- values: 'active','terminated','on_leave' or NULL
  performance_score INT -- 1..10 scale, NULL if not yet reviewed
);


In [0]:
%sql
ALTER TABLE sql.practice.employees ALTER COLUMN salary DROP NOT NULL;
ALTER TABLE sql.practice.employees ALTER COLUMN status DROP NOT NULL;

INSERT INTO sql.practice.employees (employee_id, first_name, last_name, department_id, hire_date, salary, manager_id, status, performance_score) VALUES
(101, 'Asha', 'Kumar', 1, '2018-03-15', 98000.00, 201, 'active', 8),
(102, 'Ben', 'Lopez', 2, '2020-07-01', 72000.00, 202, 'active', 7),
(103, 'Chloe', 'Nguyen', 1, '2021-11-20', NULL, 201, 'on_leave', NULL),
(104, 'Darius', 'Singh', NULL, '2019-05-25', 45000.00, NULL, 'active', 6),
(105, 'Evelyn', 'Park', 3, '2017-01-10', 65000.00, 203, 'terminated', 5),
(106, 'Felix', 'Brown', 2, '2022-09-01', 52000.00, 202, 'active', NULL),
(107, 'Greta', 'Miller', 4, '2016-12-12', 110000.00, NULL, NULL, 9),
(108, 'Hao', 'Zhang', 1, '2023-02-15', 40000.00, 101, 'active', 4),
(109, 'Ivy', 'Chen', 3, '2024-06-01', NULL, 203, 'active', NULL);

In [0]:
%sql
SELECT * from sql.practice.employees;

In [0]:
%sql
-- Orders table
CREATE TABLE IF NOT EXISTS sql.practice.orders (
  order_id INT PRIMARY KEY,
  customer_name VARCHAR(60) NOT NULL,
  order_date DATE NOT NULL,
  product VARCHAR(60) NOT NULL,
  amount DECIMAL(9,2) NOT NULL,
  shipped_date DATE,
  canceled BOOLEAN  -- TRUE if canceled, FALSE if shipped, NULL if pending
);

In [0]:
%sql
INSERT INTO sql.practice.orders (order_id, customer_name, order_date, product, amount, shipped_date, canceled) VALUES
(5001, 'Acme Corp', '2024-09-01', 'Widget A', 1250.00, '2024-09-03', FALSE),
(5002, 'Beta LLC', '2024-09-05', 'Gadget X', 780.50, NULL, NULL),
(5003, 'Gamma Inc', '2024-08-30', 'Widget A', 1250.00, '2024-09-01', FALSE),
(5004, 'Delta Co', '2024-09-07', 'Service Plan', 300.00, NULL, TRUE),
(5005, 'Acme Corp', '2024-09-10', 'Gadget X', 780.50, NULL, FALSE),
(5006, 'Echo Ltd', '2024-09-12', 'Widget B', 450.00, NULL, NULL);

In [0]:
%sql
SELECT * FROM sql.practice.orders;

In [0]:
%sql
SELECT * FROM sql.practice.employees;

In [0]:
%sql
SELECT concat(first_name,last_name) as full_name,salary from sql.practice.employees order by salary desc;

In [0]:
%sql
--Show department name and number of employees in each department (include departments with zero employees).

SELECT d.name AS department_name, count(e.employee_id) AS employee_count
FROM sql.practice.departments d
LEFT JOIN sql.practice.employees e ON d.department_id = e.department_id
GROUP BY d.department_id, d.name
ORDER BY d.department_id;

In [0]:
%sql
--For the orders table, show each distinct product and the total revenue (sum of amount) per product.
SELECT product, SUM(amount) as total_revenue FROM sql.practice.orders GROUP BY product;

In [0]:
%sql
SELECT * FROM sql.practice.employees;

In [0]:
%sql
SELECT * FROM sql.practice.employees where salary>60000 AND salary<80000;

In [0]:
%sql
SELECT * FROM sql.practice.employees where status='active' and performance_score is not null;

In [0]:
%sql
--Find employees hired between 2019-01-01 and 2022-12-31.
SELECT * FROM sql.practice.employees WHERE hire_date BETWEEN '2019-01-01' AND '2022-12-31'